# Split retail_store_inventory.csv into four component files

Produces:
- `sales_daily.csv`
- `inventory_snapshots.csv`
- `sku_master.csv`
- `calender.csv`

**Note:** In the source file, `Category` is not consistently tied to `Product ID` (each Product ID appears with all 5 categories roughly equally often), and the same goes for `Store ID` <-> `Region`, and `Date` <-> `Weather`/`Seasonality`. So this notebook does **not** fabricate a clean dimension table for those fields. Instead:
- `sku_master.csv` reports the most-common category per Product ID plus a match-rate column, so the inconsistency is visible rather than hidden.
- `calender.csv` is built purely from the `Date` value itself (Year, Month, day-of-week, etc.), not from the noisy Weather/Seasonality columns.

In [1]:
import pandas as pd

SOURCE_PATH = "retail_store_inventory.csv"  # update to your input file's path

df = pd.read_csv(SOURCE_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


## 1. `sales_daily.csv` — transactional sales facts

In [2]:
sales_daily = df[[
    'Date', 'Store ID', 'Product ID', 'Units Sold',
    'Price', 'Discount', 'Holiday/Promotion', 'Demand Forecast'
]].copy()
sales_daily['Date'] = sales_daily['Date'].dt.strftime('%Y-%m-%d')
sales_daily.to_csv('sales_daily.csv', index=False)
sales_daily.head()

,Date,Store ID,Product ID,Units Sold,Price,Discount,Holiday/Promotion,Demand Forecast
0,2022-01-01,S001,P0001,127,33.50,20,0,135.47
1,2022-01-01,S001,P0002,150,63.01,20,0,144.04
2,2022-01-01,S001,P0003,65,27.99,10,1,74.02
3,2022-01-01,S001,P0004,61,32.72,10,1,62.18
4,2022-01-01,S001,P0005,14,73.64,0,0,9.26


## 2. `inventory_snapshots.csv` — stock/replenishment facts

In [3]:
inventory_snapshots = df[[
    'Date', 'Store ID', 'Product ID', 'Inventory Level',
    'Units Ordered', 'Competitor Pricing'
]].copy()
inventory_snapshots['Date'] = inventory_snapshots['Date'].dt.strftime('%Y-%m-%d')
inventory_snapshots.to_csv('inventory_snapshots.csv', index=False)
inventory_snapshots.head()

,Date,Store ID,Product ID,Inventory Level,Units Ordered,Competitor Pricing
0,2022-01-01,S001,P0001,231,55,29.69
1,2022-01-01,S001,P0002,204,66,66.16
2,2022-01-01,S001,P0003,102,51,31.32
3,2022-01-01,S001,P0004,469,164,34.74
4,2022-01-01,S001,P0005,166,135,68.95


## 3. `sku_master.csv` — one row per Product ID

In [4]:
cat_counts = df.groupby('Product ID')['Category'].value_counts().unstack(fill_value=0)
sku_master_rows = []
for pid, row in cat_counts.iterrows():
    total = row.sum()
    top_cat = row.idxmax()
    top_pct = round(100 * row.max() / total, 1)
    observed = ";".join(sorted([c for c in row.index if row[c] > 0]))
    sku_master_rows.append({
        'Product ID': pid,
        'Most Common Category': top_cat,
        'Most Common Category %': top_pct,
        'All Categories Observed': observed
    })
sku_master = pd.DataFrame(sku_master_rows).sort_values('Product ID')
sku_master.to_csv('sku_master.csv', index=False)
sku_master.head()

,Product ID,Most Common Category,Most Common Category %,All Categories Observed
0,P0001,Groceries,20.9,Clothing;Electronics;Furniture;Groceries;Toys
1,P0002,Furniture,20.6,Clothing;Electronics;Furniture;Groceries;Toys
2,P0003,Groceries,20.8,Clothing;Electronics;Furniture;Groceries;Toys
3,P0004,Electronics,20.4,Clothing;Electronics;Furniture;Groceries;Toys
4,P0005,Groceries,20.6,Clothing;Electronics;Furniture;Groceries;Toys


## 4. `calender.csv` — pure date-derived calendar dimension

In [5]:
dates = pd.DataFrame({'Date': sorted(df['Date'].unique())})
dates['Date'] = pd.to_datetime(dates['Date'])
dates['Year'] = dates['Date'].dt.year
dates['Quarter'] = dates['Date'].dt.quarter
dates['Month'] = dates['Date'].dt.month
dates['MonthName'] = dates['Date'].dt.strftime('%B')
dates['Day'] = dates['Date'].dt.day
dates['DayOfWeek'] = dates['Date'].dt.dayofweek + 1  # 1 = Monday
dates['DayName'] = dates['Date'].dt.strftime('%A')
dates['WeekOfYear'] = dates['Date'].dt.isocalendar().week
dates['IsWeekend'] = dates['DayOfWeek'].isin([6, 7])
dates['Date'] = dates['Date'].dt.strftime('%Y-%m-%d')
dates.to_csv('calender.csv', index=False)
dates.head()

,Date,Year,Quarter,Month,MonthName,Day,DayOfWeek,DayName,WeekOfYear,IsWeekend
0,2022-01-01,2022,1,1,January,1,6,Saturday,52,True
1,2022-01-02,2022,1,1,January,2,7,Sunday,52,True
2,2022-01-03,2022,1,1,January,3,1,Monday,1,False
3,2022-01-04,2022,1,1,January,4,2,Tuesday,1,False
4,2022-01-05,2022,1,1,January,5,3,Wednesday,1,False


## Summary

In [6]:
print("sales_daily:", sales_daily.shape)
print("inventory_snapshots:", inventory_snapshots.shape)
print("sku_master:", sku_master.shape)
print("calender:", dates.shape)

sales_daily: (73100, 8)
inventory_snapshots: (73100, 6)
sku_master: (20, 4)
calender: (731, 10)
